# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GuellifTakiEddine/flyrank-intern-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane: Content Refresh (warehouse version).**

One row in my analysis represents one **(client, content item)** pair summarized over the mid-panel month **2026-03**.

The raw warehouse table `fact_content_daily_performance` has a finer grain: one row per **(report_date, client, content item)**. For this notebook, I aggregate those daily records into one monthly record per content item because content refresh decisions are made periodically rather than every day.

The queries below verify both the raw warehouse grain and the aggregated monthly grain.

In [ ]:
import duckdb, getpass
con = duckdb.connect()
token = getpass.getpass("HF token: ")  # or use Colab's Secrets panel instead of getpass
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month = "2026-03"  # mid-panel — NEVER the _sample/June file for label logic

# Full column list (untruncated) — we only saw 20/31 columns in the console view
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month={month}/*.parquet') LIMIT 1").df()
print(cols.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

These fields are used as model inputs because they are available before the refresh decision and describe the current search performance and user engagement of each webpage.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These features are all known at the decision moment and can help estimate which pages are good candidates for content refresh.

---

### Label / Proxy

The warehouse does not contain a direct "content refresh opportunity" label.

For the starter project I will construct a **proxy label** from future performance in a later notebook. The final objective is to rank webpages according to their refresh priority rather than predict a business label directly.

---

### Context

These columns provide useful context for filtering or grouping the data but are not used directly as predictive features.

- report_date
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These fields describe data availability and the observation period.

---

### Excluded

The following columns are excluded from the feature set.

- client_hash_id
- content_hash_id

These are identifiers that uniquely identify clients and webpages. They do not describe page behaviour and would not generalize to unseen data, so they should not be used for machine learning.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# Query 1 — Verify the raw grain
grain = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows
FROM read_parquet(
    '{base}/fact_content_daily_performance/month={month}/*.parquet'
)
""").df()

print("=== Query 1: Raw grain ===")
print(grain)


# Query 2 — Verify the monthly slice
summary = con.sql(f"""
SELECT
    COUNT(*) AS raw_rows,
    COUNT(DISTINCT content_hash_id) AS n_content_items,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
    '{base}/fact_content_daily_performance/month={month}/*.parquet'
)
""").df()

print("\n=== Query 2: Month summary ===")
print(summary)


# Query 3 — Availability (required: IS TRUE)
availability = con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet(
    '{base}/fact_content_daily_performance/month={month}/*.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").df()

print("\n=== Query 3: Availability ===")
print(availability)

=== Query 1: Raw grain ===
   total_rows  unique_rows
0     9841378      9841378

=== Query 2: Month summary ===
   raw_rows  n_content_items  first_day   last_day
0   9841378           331437 2026-03-01 2026-03-31

=== Query 3: Availability ===
   rows_after_filter
0             364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset contains historical Google Search Console and Google Analytics measurements, but it cannot explain why performance changes. Factors such as Google algorithm updates, competitor actions, seasonality, or editorial decisions are not directly observed.

The warehouse also does not contain a direct "content refresh opportunity" label. Any label used during model development will therefore be a proxy constructed from the available data.

Finally, this notebook focuses on a single month (2026-03). Although this is appropriate for developing the data contract, conclusions drawn from one month may not generalize to every time period.

In [8]:
print("Rows in March:", summary["raw_rows"][0])
print("Content items:", summary["n_content_items"][0])
print("Rows after availability filter:", availability["rows_after_filter"][0])

Rows in March: 9841378
Content items: 331437
Rows after availability filter: 364347


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.